# 3. Deep Learning: LSTM

Adapted from Taimour Karim's original `3_deepLearning.ipynb` (see README
Credits). **The original never learned** — training loss sat flat at
`ln(3) ≈ 1.0986` (the entropy of pure random 3-class guessing) for all 5
epochs, test accuracy 32.7%. Two independent bugs caused this, both found
by actually running the notebook rather than just reading the code:

1. `Tokenizer(num_words=500)` capped the vocabulary far below the
   `Embedding(5000, ...)` layer's declared size, so most tokens were
   out-of-vocabulary.
2. `max_length` was set to the single *longest* review in the dataset
   (711 tokens), so a typical ~34-word review ended up padded with
   ~95% zeros — and the `Embedding` layer had no `mask_zero=True`, so
   the LSTM couldn't tell real content from padding. With that much
   padding drowning out the signal, the model's predictions collapsed
   to the same near-uniform output for every input, regardless of vocab
   size.

This version matches vocabulary sizes, caps sequence length at the 95th
percentile instead of the single longest outlier, and masks padding.

In [43]:
import nltk
import numpy as np
import pandas as pd
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import LSTM, Dense, Dropout, Embedding, Input
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer

for resource in ["punkt", "punkt_tab", "stopwords"]:
    nltk.download(resource, quiet=True)

## Load data

In [44]:
df = pd.read_csv("../data/cleaned/reviews.csv")
df = df.dropna()
print(df.shape)

(19773, 6)


## Prepare sequences

In [45]:
stop_words = set(stopwords.words("english"))


def preprocess_text(text: str) -> str:
    tokens = word_tokenize(text)
    tokens = [w for w in tokens if w.isalpha() and w not in stop_words]
    return " ".join(tokens)


df["processed_comments"] = df["cleaned_comments"].apply(preprocess_text)

In [46]:
VOCAB_SIZE = 5000  # matches the Embedding layer below — this is the bug fix

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(df["processed_comments"])
sequences = tokenizer.texts_to_sequences(df["processed_comments"])

In [47]:
# max(len(seq) for seq in sequences) would pad every sequence to the
# length of the single longest review (711 tokens) - a typical ~34-word
# review would then be ~95% padding, which drowned out the real signal
# (see markdown above). Cap at the 95th percentile instead, and truncate
# the rare longer reviews.
max_length = int(np.percentile([len(seq) for seq in sequences], 95))
padded_sequences = pad_sequences(
    sequences, maxlen=max_length, padding="post", truncating="post"
)

In [48]:
labels = pd.get_dummies(df["sentiment"]).values

In [49]:
X_train, X_test, y_train, y_test = train_test_split(
    padded_sequences, labels, test_size=0.2, random_state=42
)

## Model

Back to the original author's sizing (embedding 128, LSTM 64, dense 32) -
the smaller version (64/32/16) was a CPU-speed precaution from before the
padding/masking bug was found and fixed. Now that epochs run in seconds
instead of a minute each, there's headroom to try the larger model;
`EarlyStopping` still cuts training off once validation loss stops
improving, so the larger `epochs` cap below costs nothing if it doesn't
need it. Watch validation accuracy specifically, not just training
accuracy - the risk with a bigger model on this small (~16k row)
training set is overfitting, not underfitting.

In [50]:
model = Sequential()
model.add(Input(shape=(max_length,)))
model.add(Embedding(VOCAB_SIZE, 128, mask_zero=True))
model.add(LSTM(64, dropout=0.2, recurrent_dropout=0.2))
model.add(Dense(32, activation="relu"))
model.add(Dropout(0.5))
model.add(Dense(labels.shape[1], activation="softmax"))

model.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ (None, 104, 128)       │       640,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_4 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 691,587 (2.64 MB)

 Trainable params: 691,587 (2.64 MB)

 Non-trainable params: 0 (0.00 B)

In [51]:
early_stopping = EarlyStopping(monitor="val_loss", patience=3, verbose=1, restore_best_weights=True)

history = model.fit(
    X_train,
    y_train,
    epochs=50,
    batch_size=128,
    validation_split=0.2,
    callbacks=[early_stopping],
)

Epoch 1/50
99/99 ━━━━━━━━━━━━━━━━━━━━ 12s 85ms/step - accuracy: 0.6513 - loss: 0.7949 - val_accuracy: 0.8116 - val_loss: 0.4939
Epoch 2/50
99/99 ━━━━━━━━━━━━━━━━━━━━ 7s 75ms/step - accuracy: 0.8509 - loss: 0.4305 - val_accuracy: 0.8657 - val_loss: 0.3962
Epoch 3/50
99/99 ━━━━━━━━━━━━━━━━━━━━ 7s 74ms/step - accuracy: 0.9032 - loss: 0.3066 - val_accuracy: 0.8932 - val_loss: 0.3183
Epoch 4/50
99/99 ━━━━━━━━━━━━━━━━━━━━ 7s 73ms/step - accuracy: 0.9241 - loss: 0.2559 - val_accuracy: 0.8824 - val_loss: 0.3533
Epoch 5/50
99/99 ━━━━━━━━━━━━━━━━━━━━ 7s 74ms/step - accuracy: 0.9390 - loss: 0.2059 - val_accuracy: 0.8979 - val_loss: 0.3196
Epoch 6/50
99/99 ━━━━━━━━━━━━━━━━━━━━ 7s 75ms/step - accuracy: 0.9443 - loss: 0.1816 - val_accuracy: 0.8951 - val_loss: 0.3194
Epoch 6: early stopping
Restoring model weights from the end of the best epoch: 3.


## Evaluate

Sanity check: with 3 balanced classes, a model that learned nothing would
plateau at ~33% accuracy and a flat loss around `ln(3) ≈ 1.099` — exactly
what the original, buggy notebook showed. This version should clearly beat
that baseline.

In [ ]:
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Test Loss: {loss:.4f}  (random-guess baseline: {np.log(3):.4f})")
print(f"Test Accuracy: {accuracy * 100:.2f}%  (random-guess baseline: 33.3%)")
assert accuracy > 0.40, "Model is not clearly beating the random-guess baseline — investigate before proceeding"

124/124 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.8812 - loss: 0.3596
Test Loss: 0.3596  (random-guess baseline: 1.0986)
Test Accuracy: 88.12%  (random-guess baseline: 33.3%)


: 